# Train BPR

In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [4]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [8]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "category"] 
    },
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [9]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
21 Jun 13:22    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
21 Jun 13:22    INFO  [Evaluation]: eval_batch_size = [409600000] eval_args: [{'split': None, 'order': 'TO', 'group_by': 'user', 'mode': {'valid':

## Train BPR

In [10]:
model: BPR = BPR(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
21 Jun 13:22    INFO  epoch 0 training [time: 15.80s, train loss: 754.5382]
21 Jun 13:22    INFO  epoch 0 evaluating [time: 10.97s, valid_score: 0.003600]
21 Jun 13:22    INFO  valid result: 
ndcg@20 : 0.0036    recall@20 : 0.0063    mrr@20 : 0.0042
21 Jun 13:22    INFO  Saving current: saved/BPR-Jun-21-2026_13-22-13.pth
21 Jun 13:22    INFO  epoch 1 training [time: 12.95s, train loss: 631.8274]
21 Jun 13:23    INFO  epoch 1 evaluating [time: 11.91s, valid_score: 0.005900]
21 Jun 13:23    INFO  valid result: 
ndcg@20 : 0.0059    recall@20 : 0.0104    mrr@20 : 0.0071
21 Jun 13:23    INFO  Saving current: saved/BPR-Jun-21-2026_13-22-13.pth
21 Jun 13:23    INFO  epoch 2 training [time: 12.95s, train lo

In [11]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0071
Best valid result:
  ndcg@20: 0.0071
  recall@20: 0.0127
  mrr@20: 0.0085


## Evaluate on test set

In [14]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

21 Jun 13:30    INFO  Loading model structure and parameters from saved/BPR-Jun-21-2026_13-22-13.pth


Test results (Overall):
  ndcg@20: 0.0068
  recall@20: 0.0116
  mrr@20: 0.0085


In [15]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)

21 Jun 13:31    INFO  Loading model structure and parameters from saved/BPR-Jun-21-2026_13-22-13.pth



Evaluation (warm)
--------------------
  Interactions: 46742
  ndcg@20: 0.0058
  recall@20: 0.0092
  mrr@20: 0.0094


21 Jun 13:31    INFO  Loading model structure and parameters from saved/BPR-Jun-21-2026_13-22-13.pth



Evaluation (cold)
--------------------
  Interactions: 150964
  ndcg@20: 0.0070
  recall@20: 0.0120
  mrr@20: 0.0083
